# 02 Mesh EPC, Transport, And Mobility

This notebook computes mesh EPC and then estimates linewidth, SERTA transport, and SI mobility using DeePTB APIs. It uses the matsci-0-3 SKF model and dftbephy Graphene phonon inputs by default.

In [ ]:
from pathlib import Path
import json
import numpy as np

from dptb.nn.dftbsk import DFTBSK
from dptb.postprocess.unified import TBSystem
from dptb.postprocess.unified.eph import Phonons
import importlib
import epc_notebook_utils
importlib.reload(epc_notebook_utils)
from epc_notebook_utils import external_path, regularize_tiny_negative_frequencies

ROOT = Path.cwd()
WORK = ROOT / "work"
WORK.mkdir(exist_ok=True)

# Set these environment variables before running the notebooks.
DFTBEPHY_GRAPHENE = external_path(
    "DEEPTB_EPC_DFTBEPHY_GRAPHENE",
    "the dftbephy Graphene example directory, e.g. /path/to/dftbephy/examples/Graphene",
    candidates=(
        Path.cwd().parent.parent / "dftbephy" / "examples" / "Graphene",
        Path.cwd().parent / "dftbephy" / "examples" / "Graphene",
        Path("/Users/aisiqg/Desktop/work/github/dftbephy/examples/Graphene"),
    ),
)
MATSCI_SK = external_path(
    "DEEPTB_EPC_MATSCI_SK",
    "the matsci-0-3 SKF directory containing C-C.skf",
    candidates=(
        Path.cwd().parent.parent / "matsci-0-3",
        Path.cwd().parent / "matsci-0-3",
        Path("/Users/aisiqg/Desktop/work/github/matsci-0-3"),
    ),
)

STRUCTURE = ROOT / "graphene.vasp"
if not STRUCTURE.exists():
    STRUCTURE = DFTBEPHY_GRAPHENE / "opt" / "geo_end.gen"

PHONOPY_YAML = DFTBEPHY_GRAPHENE / "phonons" / "phonopy_disp.yaml"
FORCE_SETS = DFTBEPHY_GRAPHENE / "phonons" / "FORCE_SETS"

# matsci-0-3 C-C.skf contains the carbon SK data used by this example model.
BASIS = {"C": ["2s", "2p"]}

for path in [MATSCI_SK, STRUCTURE, PHONOPY_YAML, FORCE_SETS]:
    if not path.exists():
        raise FileNotFoundError(path)

model = DFTBSK(
    basis=BASIS,
    skdata=str(MATSCI_SK),
    overlap=True,
    dtype="float64",
    device="cpu",
    interp_method="smooth_intp",
    r_max=6.0,
)
system = TBSystem(data=str(STRUCTURE), calculator=model)
print(system.atoms)
print(system.model.name, system.model.basis)


In [ ]:
from dptb.postprocess.unified.eph import (
    LinewidthData,
    Phonons,
    EPCMeshSpec,
    EPCMeshData,
    load_epc_mesh_chunked_artifact,
    compute_linewidth_mesh,
    compute_linewidth_mesh_chunked_artifact,
    compute_relaxation_time_mesh,
    compute_serta_transport_from_epc,
    compute_serta_mobility_si,
    compute_serta_mobility_scan_si,
    compute_band_velocities_hamiltonian_derivative,
)
from epc_notebook_utils import require_file

phonons_mesh_npz = require_file(
    WORK / "phonons_mesh.npz",
    "Run 00_prepare_external_phonons.ipynb first to generate phonons_mesh.npz.",
)
phonons = regularize_tiny_negative_frequencies(Phonons.load_npz(phonons_mesh_npz))
bands = [0, 1, 2, 3]
chemical_potential = 0.0
temperature = 0.025
sigma = 0.01
spin_degeneracy = 2


## Full in-memory small mesh

Use this for small tests and debugging. For large meshes, prefer the chunked artifact cell below.

In [ ]:
mesh_spec = EPCMeshSpec(
    k_mesh=[2, 2, 1],
    time_reversal=True,
)

epc_mesh = system.eph.compute_mesh(
    mesh_spec=mesh_spec,
    phonons=phonons,
    bands=bands,
    displacement=1e-3,
    output_npz=WORK / "epc_mesh_data.npz",
)
print(epc_mesh.coupling_strength.shape)
print(epc_mesh.kpoint_weights, epc_mesh.qpoint_weights)


## Mesh linewidth and relaxation time

In [ ]:
mesh_linewidth = compute_linewidth_mesh(
    epc_mesh,
    chemical_potential=chemical_potential,
    temperature=temperature,
    sigma=sigma,
    broadening="gaussian",
    mode_resolved=True,
)
mesh_linewidth.save_npz(WORK / "mesh_linewidth.npz")

mesh_tau = compute_relaxation_time_mesh(mesh_linewidth, sum_modes=True)
mesh_tau.save_npz(WORK / "mesh_relaxation_time.npz")
print(mesh_linewidth.linewidth.shape, mesh_tau.relaxation_time.shape)


## Transport and SI mobility

`TransportData` uses DeePTB internal fractional-k units. `MobilityData` converts velocities to SI using the reciprocal cell.

In [ ]:
fractional_velocities = compute_band_velocities_hamiltonian_derivative(
    system=system,
    kpoints=epc_mesh.kpoints,
    bands=epc_mesh.band_indices,
)

transport_linewidth = LinewidthData(
    linewidth=mesh_linewidth.linewidth.sum(axis=-1),
    absorption=mesh_linewidth.absorption.sum(axis=-1),
    emission=mesh_linewidth.emission.sum(axis=-1),
    metadata={"source": "mesh_linewidth_summed_for_transport", "mesh_linewidth_schema": mesh_linewidth.metadata.get("schema"), "linewidth_unit": mesh_linewidth.metadata.get("linewidth_unit", "eV")},
)

transport = compute_serta_transport_from_epc(
    system=system,
    epc_data=epc_mesh.epc_data,
    linewidth_data=transport_linewidth,
    chemical_potential=chemical_potential,
    temperature=temperature,
    kpoint_weights=epc_mesh.kpoint_weights,
    spin_degeneracy=spin_degeneracy,
    volume=1.0,
    velocity_source="hamiltonian_derivative",
)
transport.save_npz(WORK / "transport.npz")
print("transport conductivity", transport.conductivity)
print("transport metadata", transport.metadata)

reciprocal_cell = 2.0 * np.pi * system.atoms.cell.reciprocal()
mobility = compute_serta_mobility_si(
    eigenvalues=epc_mesh.eigenvalues_k,
    velocities=fractional_velocities,
    linewidth=mesh_linewidth.linewidth.sum(axis=-1),
    chemical_potential=chemical_potential,
    temperature=temperature,
    reciprocal_cell=reciprocal_cell,
    kpoint_weights=epc_mesh.kpoint_weights,
    spin_degeneracy=spin_degeneracy,
    dimension="2d",
    area=float(np.linalg.norm(np.cross(system.atoms.cell[0], system.atoms.cell[1]))),
)
mobility.save_npz(WORK / "mobility.npz")
print("mobility", mobility.mobility)
print("carrier density", mobility.carrier_density)


## Chunked artifact path

This is the large-mesh friendly API. It writes one q or k chunk at a time. Execution is still serial.

In [ ]:
artifact_dir = WORK / "epc_mesh_artifact"
artifact_manifest = system.eph.compute_mesh_chunked_artifact(
    mesh_spec=EPCMeshSpec(k_mesh=[2, 2, 1], q_chunk_size=1, time_reversal=True),
    phonons=phonons,
    directory=artifact_dir,
    axis="q",
    bands=bands,
    displacement=1e-3,
)
print(json.dumps(artifact_manifest, indent=2)[:2000])

artifact_linewidth = compute_linewidth_mesh_chunked_artifact(
    artifact_dir,
    chemical_potential=chemical_potential,
    temperature=temperature,
    sigma=sigma,
    broadening="gaussian",
)
artifact_linewidth.save_npz(WORK / "mesh_linewidth_from_artifact.npz")
print(artifact_linewidth.linewidth.shape)


## Mobility scan with fixed linewidth

In [ ]:
scan = compute_serta_mobility_scan_si(
    eigenvalues=epc_mesh.eigenvalues_k,
    velocities=fractional_velocities,
    linewidth=mesh_linewidth.linewidth.sum(axis=-1),
    chemical_potentials=np.array([-0.05, 0.0, 0.05]),
    temperatures=np.array([0.015, 0.025, 0.050]),
    reciprocal_cell=reciprocal_cell,
    kpoint_weights=epc_mesh.kpoint_weights,
    spin_degeneracy=spin_degeneracy,
    dimension="2d",
    area=float(np.linalg.norm(np.cross(system.atoms.cell[0], system.atoms.cell[1]))),
)
scan.save_npz(WORK / "mobility_scan.npz")
print(scan.mobility.shape)
print(scan.metadata)
